# LightGBM Alzheimer's Disease Classification Pipeline

**Heavy, highly-optimized LightGBM model for AD vs MCI vs CN prediction**

This notebook implements a complete classification pipeline with:
- Optuna hyperparameter optimization (60 trials)
- Comprehensive evaluation metrics
- Advanced visualizations (confusion matrix, feature importance, SHAP, ROC curves)
- GPU support (automatic detection)

---

## 1. Install Dependencies

In [ ]:
!pip install -q lightgbm optuna shap pandas openpyxl scikit-learn matplotlib seaborn

## 2. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# LightGBM
import lightgbm as lgb
from lightgbm import LGBMClassifier

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# Optuna
import optuna
from optuna.samplers import TPESampler

# SHAP
import shap

# Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✓ All libraries imported successfully!")

## 3. Configuration

In [ ]:
# Feature columns
FEATURE_COLUMNS = [
    'csf_abeta40_value', 'csf_abeta42_value', 'csf_nfl_value', 'csf_ptau_value',
    'csf_tau_value', 'formic_acid_value', 'lactoferrin_value',
    'dha_value', 'ntp_value', 'mmse_score'
]

TARGET_COLUMN = 'diagnosis'
TEST_SIZE = 0.2
N_OPTUNA_TRIALS = 60
CV_FOLDS = 5
RESULTS_DIR = './results'

print(f"✓ Configuration set:")
print(f"  - Features: {len(FEATURE_COLUMNS)}")
print(f"  - Test size: {TEST_SIZE*100:.0f}%")
print(f"  - Optuna trials: {N_OPTUNA_TRIALS}")
print(f"  - CV folds: {CV_FOLDS}")

## 4. Upload Data

Upload your Excel file (.xlsx) containing the dataset.

In [ ]:
from google.colab import files

print("Please upload your Excel file...")
uploaded = files.upload()
data_file = list(uploaded.keys())[0]
print(f"\n✓ File uploaded: {data_file}")

## 5. Load and Preprocess Data

In [ ]:
def load_and_preprocess_data(file_path):
    print("=" * 70)
    print("LOADING DATA")
    print("=" * 70)
    
    # Load Excel file
    df = pd.read_excel(file_path)
    print(f"✓ Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Select features and target
    required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing_cols = set(required_columns) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing columns in dataset: {missing_cols}")
    
    df_filtered = df[required_columns].copy()
    print(f"✓ Selected {len(FEATURE_COLUMNS)} features + 1 target column")
    
    # Drop rows with missing values
    initial_rows = len(df_filtered)
    df_filtered = df_filtered.dropna()
    dropped_rows = initial_rows - len(df_filtered)
    print(f"✓ Dropped {dropped_rows} rows with missing values ({len(df_filtered)} remaining)")
    
    # Check class distribution
    print(f"\n--- Target Distribution ---")
    class_counts = df_filtered[TARGET_COLUMN].value_counts()
    for label, count in class_counts.items():
        print(f"  {label}: {count} ({count/len(df_filtered)*100:.1f}%)")
    
    # Encode target labels
    X = df_filtered[FEATURE_COLUMNS].values
    y = df_filtered[TARGET_COLUMN].values
    
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    print(f"\n✓ Encoded labels: {dict(enumerate(label_encoder.classes_))}")
    
    # Train-test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=y_encoded
    )
    
    print(f"\n✓ Train-test split (stratified):")
    print(f"  Training set: {X_train.shape[0]} samples")
    print(f"  Test set: {X_test.shape[0]} samples")
    print("=" * 70)
    
    return X_train, X_test, y_train, y_test, label_encoder, FEATURE_COLUMNS

# Load data
X_train, X_test, y_train, y_test, label_encoder, feature_names = load_and_preprocess_data(data_file)
n_classes = len(label_encoder.classes_)

## 6. Hyperparameter Optimization with Optuna

This will take several minutes. Progress bar will be displayed.

In [ ]:
def objective(trial, X_train, y_train, n_classes):
    params = {
        'objective': 'multiclass',
        'num_class': n_classes,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': RANDOM_SEED,
        'device': 'gpu' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu',
        
        # Tree structure
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 12, 30),
        
        # Learning parameters
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.05, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000, step=100),
        
        # Regularization
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 1e-1, log=True),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
        
        # Sampling
        'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        
        # Advanced parameters
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 1.0),
    }
    
    model = LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    
    return scores.mean()

# Run optimization
print("=" * 70)
print("HYPERPARAMETER OPTIMIZATION (OPTUNA)")
print("=" * 70)
print(f"Running {N_OPTUNA_TRIALS} trials with {CV_FOLDS}-fold cross-validation...")
print("This may take several minutes...\n")

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_SEED)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train, n_classes),
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
    n_jobs=1
)

print("\n✓ Optimization complete!")
print(f"  Best CV Accuracy: {study.best_value:.4f}")
print(f"  Best Trial: #{study.best_trial.number}")

# Construct best parameters
best_params = study.best_params
best_params.update({
    'objective': 'multiclass',
    'num_class': n_classes,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': RANDOM_SEED,
    'device': 'gpu' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu',
})

print("\n--- Best Parameters ---")
for key, value in sorted(best_params.items()):
    if key not in ['objective', 'num_class', 'metric', 'boosting_type', 'verbosity', 'random_state', 'device']:
        print(f"  {key}: {value}")
print("=" * 70)

best_cv_score = study.best_value

## 7. Train Final Model

In [ ]:
print("=" * 70)
print("TRAINING FINAL MODEL")
print("=" * 70)

model = LGBMClassifier(**best_params)
model.fit(X_train, y_train)

print("✓ Model training complete!")
print("=" * 70)

## 8. Evaluate Model

In [ ]:
def calculate_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred)
    specificity = {}
    
    for i in range(n_classes):
        tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
        fp = np.sum(cm[:, i]) - cm[i, i]
        specificity[i] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return specificity

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("=" * 70)
print("### Performance Metrics")
print("=" * 70)

# Basic metrics
accuracy = accuracy_score(y_test, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average='macro'
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted'
)

print(f"\nAccuracy: {accuracy:.2f}")
print(f"Macro Avg → Precision: {precision_macro:.2f} | Recall: {recall_macro:.2f} | F1: {f1_macro:.2f}")
print(f"Weighted Avg → Precision: {precision_weighted:.2f} | Recall: {recall_weighted:.2f} | F1: {f1_weighted:.2f}")

# Per-class metrics
print("\n--- Per-Class ---")
precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
    y_test, y_pred, average=None
)

class_names = label_encoder.classes_
specificity_per_class = calculate_specificity(y_test, y_pred, len(class_names))

for i, class_name in enumerate(class_names):
    print(f"{class_name}: Precision {precision_per_class[i]:.2f} | "
          f"Recall {recall_per_class[i]:.2f} | F1 {f1_per_class[i]:.2f}")

# Confusion Matrix
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
header = "           " + "".join([f"{name:>7}" for name in class_names])
print(header)
for i, class_name in enumerate(class_names):
    row = f"    {class_name:>2}    " + "".join([f"{cm[i][j]:>7}" for j in range(len(class_names))])
    print(row)

# ROC-AUC Metrics
print("\n--- ROC-AUC Metrics ---")
y_test_bin = label_binarize(y_test, classes=range(len(class_names)))

for i, class_name in enumerate(class_names):
    auc_score = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
    print(f"{class_name} AUC: {auc_score:.2f}")

auc_micro = roc_auc_score(y_test_bin, y_pred_proba, average='micro')
auc_macro = roc_auc_score(y_test_bin, y_pred_proba, average='macro')
auc_weighted = roc_auc_score(y_test_bin, y_pred_proba, average='weighted')

print(f"Micro-average AUC: {auc_micro:.2f}")
print(f"Macro-average AUC: {auc_macro:.2f}")
print(f"Weighted-average AUC: {auc_weighted:.2f}")

# Specificity and Sensitivity
print("\n--- Sensitivity & Specificity per Class ---")
for i, class_name in enumerate(class_names):
    sensitivity = recall_per_class[i]
    specificity = specificity_per_class[i]
    print(f"{class_name}: Sensitivity {sensitivity:.2f} | Specificity {specificity:.2f}")

# Cross-Validation Summary
print("\n--- Cross-Validation Summary ---")
print(f"Mean CV Accuracy: {best_cv_score:.4f}")

print("\n--- Best Hyperparameters ---")
important_params = ['num_leaves', 'max_depth', 'learning_rate', 'n_estimators',
                   'lambda_l1', 'lambda_l2', 'feature_fraction', 'bagging_fraction']
for param in important_params:
    if param in best_params:
        print(f"  {param}: {best_params[param]}")

print("=" * 70)

## 9. Visualizations

In [ ]:
# Create results directory
os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 70)
print("GENERATING VISUALIZATIONS")
print("=" * 70)

### 9.1 Confusion Matrix

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved confusion matrix")

### 9.2 Feature Importance

In [ ]:
importance = model.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance (LightGBM)', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved feature importance")

# Save CSV
importance_df.to_csv(os.path.join(RESULTS_DIR, 'feature_importance.csv'), index=False)
print("✓ Saved feature importance CSV")

### 9.3 SHAP Summary Plot

In [ ]:
try:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    plt.figure(figsize=(12, 8))
    if isinstance(shap_values, list):
        shap_values_combined = np.abs(shap_values).mean(axis=0)
        shap.summary_plot(shap_values_combined, X_test,
                        feature_names=feature_names,
                        show=False)
    else:
        shap.summary_plot(shap_values, X_test,
                        feature_names=feature_names,
                        show=False)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'shap_summary.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved SHAP summary plot")
except Exception as e:
    print(f"⚠ Warning: Could not generate SHAP plot: {str(e)}")

### 9.4 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))

colors = ['blue', 'red', 'green', 'orange', 'purple']
for i, (class_name, color) in enumerate(zip(class_names, colors[:len(class_names)])):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
            label=f'{class_name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves (One-vs-Rest)', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved ROC curves")

print("=" * 70)

## 10. Save Results

In [ ]:
print("=" * 70)
print("SAVING RESULTS")
print("=" * 70)

# Save performance metrics
metrics_path = os.path.join(RESULTS_DIR, 'performance_metrics.txt')
with open(metrics_path, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("### Performance Metrics\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Accuracy: {accuracy:.2f}\n")
    f.write(f"Macro Avg → Precision: {precision_macro:.2f} | Recall: {recall_macro:.2f} | F1: {f1_macro:.2f}\n")
    f.write(f"Weighted Avg → Precision: {precision_weighted:.2f} | Recall: {recall_weighted:.2f} | F1: {f1_weighted:.2f}\n\n")
    f.write("--- Per-Class ---\n")
    for i, class_name in enumerate(class_names):
        f.write(f"{class_name}: Precision {precision_per_class[i]:.2f} | Recall {recall_per_class[i]:.2f} | F1 {f1_per_class[i]:.2f}\n")
    f.write("\n--- Confusion Matrix ---\n")
    f.write(header + "\n")
    for i, class_name in enumerate(class_names):
        row = f"    {class_name:>2}    " + "".join([f"{cm[i][j]:>7}" for j in range(len(class_names))])
        f.write(row + "\n")
    f.write("\n--- ROC-AUC Metrics ---\n")
    for i, class_name in enumerate(class_names):
        auc_score = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
        f.write(f"{class_name} AUC: {auc_score:.2f}\n")
    f.write(f"Micro-average AUC: {auc_micro:.2f}\n")
    f.write(f"Macro-average AUC: {auc_macro:.2f}\n")
    f.write(f"Weighted-average AUC: {auc_weighted:.2f}\n")
    f.write("\n--- Sensitivity & Specificity per Class ---\n")
    for i, class_name in enumerate(class_names):
        f.write(f"{class_name}: Sensitivity {recall_per_class[i]:.2f} | Specificity {specificity_per_class[i]:.2f}\n")
    f.write("\n--- Cross-Validation Summary ---\n")
    f.write(f"Mean CV Accuracy: {best_cv_score:.4f}\n")
    f.write("\n--- Best Parameters ---\n")
    for key, value in sorted(best_params.items()):
        if key not in ['objective', 'num_class', 'metric', 'boosting_type', 'verbosity', 'random_state', 'device']:
            f.write(f"  {key}: {value}\n")
    f.write("=" * 70 + "\n")

print(f"✓ Saved performance metrics: {metrics_path}")

# Save classification report
report_path = os.path.join(RESULTS_DIR, 'classification_report.txt')
report = classification_report(y_test, y_pred, target_names=class_names)
with open(report_path, 'w') as f:
    f.write("Classification Report\n")
    f.write("=" * 70 + "\n\n")
    f.write(report)

print(f"✓ Saved classification report: {report_path}")
print("=" * 70)

## 11. Download Results

Download all results as a zip file.

In [ ]:
from google.colab import files
import shutil

# Create zip file
shutil.make_archive('results', 'zip', RESULTS_DIR)

# Download
files.download('results.zip')
print("✓ Results downloaded!")

## 12. Final Summary

In [ ]:
print("\n" + "=" * 70)
print(" PIPELINE COMPLETE!")
print("=" * 70)
print(f"\n✓ All results saved to: {RESULTS_DIR}/")
print(f"✓ Test Accuracy: {accuracy:.2f}")
print(f"✓ Macro F1-Score: {f1_macro:.2f}")
print(f"✓ Macro AUC: {auc_macro:.2f}")
print(f"✓ Best CV Accuracy: {best_cv_score:.4f}")
print("\n" + "=" * 70)
print("Results files:")
print("  - performance_metrics.txt")
print("  - classification_report.txt")
print("  - feature_importance.csv")
print("  - confusion_matrix.png")
print("  - feature_importance.png")
print("  - shap_summary.png")
print("  - roc_curves.png")
print("=" * 70 + "\n")